# Did LEVIATHAN filter the inversions?
So it doesn't seem like repeat-regions are too responsible for the false negatives (uncalled inversions). The next place to look is the `.candidates` files LEVIATHAN outputs. These files contain the list of all putatively called structural variants, which the program then filters based on some criteria to output the final called variant set. The question we're trying to address here is: **did LEVIATHAN identify the inversions initially and then filter them out?**

In [18]:
library(dplyr)
library(tidyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## Processing the candidates
I think these data have to get processed similarly to how we did the matchy-matchy for the SVs leviathan ultimately called. It may be easier to load in the big table that process produced and add a boolean column to it that's essentially "did leviaithan initially catch this?"

In [22]:
assessed_inversions <- read.csv("assess_called_sv/linkedread.sv.assessment", header = T)
assessed_inversions$candidate <- FALSE
head(assessed_inversions)

,contig,inversion,id,depth,sample,size,simulated,zygosity,assessment,position_start,position_end,technology,candidate
,<chr>,<int>,<int>,<dbl>,<int>,<chr>,<lgl>,<chr>,<chr>,<int>,<int>,<chr>,<lgl>
1,2L,1,1,0.5,11,small,TRUE,hom,false negative,3193196,3214221,linkedread,FALSE
2,2L,1,1,2.0,11,small,TRUE,hom,true positive,3193196,3214221,linkedread,FALSE
3,2L,1,1,5.0,11,small,TRUE,hom,true positive,3193196,3214221,linkedread,FALSE
4,2L,1,1,10.0,11,small,TRUE,hom,true positive,3193196,3214221,linkedread,FALSE
5,2L,1,1,20.0,11,small,TRUE,hom,true positive,3193196,3214221,linkedread,FALSE
6,2L,1,1,0.5,1,small,TRUE,hom,false negative,3193196,3214221,linkedread,FALSE


This is a simple function to generate the path to the leviathan candidates file we would be interested in parsing. It's here as a separate function to make the main processing loop less bloated and easier to read.

In [23]:
generate_candidatefile <- function(.size, .depth, .sample) {
    if (.sample <= 10){
        .cand_file <- sprintf("simulated_data/called_sv/leviathan/%s/depth_%s/by_sample90/logs/leviathan/sample_%02d.candidates", .size, .depth, .sample)
    } else {
        .cand_file <- sprintf("simulated_data/called_sv/leviathan/%s/depth_%s/by_pop90/logs/leviathan/pop1.candidates", .size, .depth)
    }
    return (.cand_file)
}

In [25]:
sizes = c("small", "medium", "large", "xl")
depths = c("0.5", "2", "5", "10", "20")

for (.size in sizes) {
    for (.depth in depths) {
        for (.sample in 1:11) {
            .assess <- filter(
                assessed_inversions,
                depth == as.double(.depth),
                size == .size,
                sample == .sample
            )
            .assess$candidate <- FALSE

            .candidates <- read.table(
                generate_candidatefile(.size, .depth, .sample),
                col.names = c("contig", "position_start", "position_end", "contig2", "position_start2", "position_end2", "barcodes")
            )
            for (i in 1:nrow(.assess)) {
                .row <- .assess[i,]
                query <- which(
                    .candidates$contig == .row$contig &
                    (.candidates$position_start - 50 <= .row$position_start & .candidates$position_end + 50 >= .row$position_end) | (.candidates$position_start2 - 50 <= .row$position_start & .candidates$position_end2 + 50 >= .row$position_end)
                )
                if(length(query) > 0){
                    .assess$candidate[i] <- TRUE
                    print("HELL YEAH, BROTHER")
                }
            }

        }
    }
}
